# Week 01 - Crawl SafeguardIoT (Depth 3) -> Markdown + JSONL

**Objective**: Crawl http://localhost:8081/ to max depth 3, save as Markdown files and consolidated JSONL corpus.

**Architecture**: Uses `SafeguardIoTWebCrawler` service from `context_engineering.application.ingest_documents_service.web_crawler`

**Provider Support**: Supports OpenRouter (multi-provider) or direct OpenAI API via `.env` configuration


In [1]:
# Cell 1: Setup & Installations
import sys

if "google.colab" in sys.modules or True:
    print("📦 Installing required packages...")
    %pip install -q playwright>=1.40.0 python-dotenv>=1.0.0 beautifulsoup4>=4.12.0 markdownify>=0.11.6 nest-asyncio>=1.5.0
    
    # Install Playwright browsers
    print("🌐 Installing Playwright browsers...")
    import subprocess
    subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True, capture_output=True)

print("✅ Packages ready")

📦 Installing required packages...
Note: you may need to restart the kernel to use updated packages.
🌐 Installing Playwright browsers...


e:\Sliit\Y4\IOT\chat_agent\.venv\Scripts\python.exe: No module named pip


✅ Packages ready


In [2]:
# Cell 2: Imports & Environment Setup
import os
import sys
import json
import time
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
import nest_asyncio

# Enable nested asyncio
nest_asyncio.apply()

# Fix for Windows asyncio subprocess issues with Playwright
import asyncio
if hasattr(asyncio, 'WindowsSelectorEventLoopPolicy'):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

provider = "OpenRouter" if openrouter_key else "OpenAI"
print("✅ Environment loaded")
print(f"🌐 Provider: {provider}")
print(f"📁 Project root: {project_root}")

✅ Environment loaded
🌐 Provider: OpenAI
📁 Project root: e:\Sliit\Y4\IOT\chat_agent


In [3]:
# Cell 3: Load Configuration
from context_engineering.config import (
    validate, dump, CRAWL_OUT_DIR
)

# Validate and display config
try:
    validate()
    dump()
except Exception as e:
    print(f"[WARN] Config note: {e}")

# Use a site-specific markdown output directory for this crawl
MARKDOWN_DIR = CRAWL_OUT_DIR / "safeguardiot_markdown"

# Ensure directories exist
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

print()
print("[OK] Output directories ready:")
print(f"   - Markdown: {MARKDOWN_DIR}")
print(f"   - JSONL: {CRAWL_OUT_DIR}")


e:\Sliit\Y4\IOT\chat_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



CONFIGURATION (NON-SECRETS ONLY)

🌐 Provider:
   Provider: openai
   Model Tier: general
   Chat Model: gpt-4o-mini
   Embedding Model: text-embedding-3-large

📁 Directories:
   Data Root: e:\Sliit\Y4\IOT\chat_agent\data
   Vector Store: e:\Sliit\Y4\IOT\chat_agent\data\vectorstore
   Markdown: e:\Sliit\Y4\IOT\chat_agent\data\nawaloka_markdown
   Cache: e:\Sliit\Y4\IOT\chat_agent\data\cag_cache

🔧 Chunking:
   Fixed Size: 800 tokens
   Fixed Overlap: 100 tokens
   Sliding Window: 512 tokens
   Sliding Stride: 256 tokens
   Parent-Child: 250 → 1200 tokens
   Late Chunk: 1000 → 300 tokens

🔍 Retrieval:
   Top-K Results: 4
   Similarity Threshold: 0.7

💾 CAG:
   Cache TTL: 86400s
   Max Cache Size: 1000

🎯 CRAG:
   Confidence Threshold: 0.6
   Expanded K: 8



[OK] Output directories ready:
   - Markdown: e:\Sliit\Y4\IOT\chat_agent\data\safeguardiot_markdown
   - JSONL: e:\Sliit\Y4\IOT\chat_agent\data


## Import Crawler Service

Using `SafeguardIoTWebCrawler` from application layer (NOT defined here!)


In [4]:
# Cell 4: Import Web Crawler Service
import importlib
from context_engineering.application.ingest_documents_service import web_crawler as web_crawler_module

importlib.reload(web_crawler_module)
SafeguardIoTWebCrawler = web_crawler_module.SafeguardIoTWebCrawler

print("[OK] SafeguardIoTWebCrawler loaded from service layer")
print("[PATH] context_engineering.application.ingest_documents_service.web_crawler")


[OK] SafeguardIoTWebCrawler loaded from service layer
[PATH] context_engineering.application.ingest_documents_service.web_crawler


## Crawl Configuration

In [5]:
# Cell 5: Crawl Configuration
BASE_URL = "http://localhost:8081/"

# Start from the homepage and discover internal links via BFS
START_URLS = [BASE_URL]

LOGIN_EMAIL = os.getenv("CRAWL_LOGIN_EMAIL", "admin@safeguard.io")
LOGIN_PASSWORD = os.getenv("CRAWL_LOGIN_PASSWORD", "Admin@123")
LOGIN_URL = BASE_URL

EXCLUDE_PATTERNS = [
    "/login", "/terms", "/privacy", "/admin",
    "/images/", "/downloads/", "/media/"
]

MAX_DEPTH = 3
REQUEST_DELAY = 1.0
JSONL_PATH = CRAWL_OUT_DIR / "safeguardiot_docs.jsonl"

print(f"[INFO] Crawl config:")
print(f"   - Base URL: {BASE_URL}")
print(f"   - Start URLs: {len(START_URLS)}")
print(f"   - Login URL: {LOGIN_URL}")
print(f"   - Login User: {LOGIN_EMAIL}")
print(f"   - Max depth: {MAX_DEPTH}")
print(f"   - Request delay: {REQUEST_DELAY}s")


[INFO] Crawl config:
   - Base URL: http://localhost:8081/
   - Start URLs: 1
   - Login URL: http://localhost:8081/
   - Login User: admin@safeguard.io
   - Max depth: 3
   - Request delay: 1.0s


## Execute Crawl

In [6]:
# Cell 6: Run Crawl with Service
start_time = time.time()

# Initialize crawler service
crawler = SafeguardIoTWebCrawler(
    base_url=BASE_URL,
    max_depth=MAX_DEPTH,
    exclude_patterns=EXCLUDE_PATTERNS,
    login_email=LOGIN_EMAIL,
    login_password=LOGIN_PASSWORD,
    login_url=LOGIN_URL
)

print()
print(f"[START] Starting crawl at {time.strftime('%H:%M:%S')}")
print()
documents = crawler.crawl(START_URLS, request_delay=REQUEST_DELAY)

elapsed = time.time() - start_time
print()
print(f"[OK] Crawl complete in {elapsed:.1f}s")
print(f"[DOCS] Documents collected: {len(documents)}")
print(f"[URLS] URLs visited: {len(crawler.visited)}")



[START] Starting crawl at 05:40:34

[AUTH] Logging in at http://localhost:8081/
[AUTH] Authenticated, starting crawl from http://localhost:8081/dashboard
[CRAWL:0] http://localhost:8081/dashboard
   [OK] Saved (453 chars, 7 links found)
   [QUEUE] Added 7 new URLs (depth 1)
   [PROGRESS] 1 saved, 1 visited, 7 queued
[CRAWL:1] http://localhost:8081/workers
   [OK] Saved (496 chars, 7 links found)
   [PROGRESS] 2 saved, 2 visited, 6 queued
[CRAWL:1] http://localhost:8081/alerts
   [OK] Saved (213 chars, 7 links found)
   [PROGRESS] 3 saved, 3 visited, 5 queued
[CRAWL:1] http://localhost:8081/settings
   [OK] Saved (1316 chars, 7 links found)
   [PROGRESS] 4 saved, 4 visited, 4 queued
[CRAWL:1] http://localhost:8081/ai-insights
   [OK] Saved (2527 chars, 7 links found)
   [PROGRESS] 5 saved, 5 visited, 3 queued
[CRAWL:1] http://localhost:8081/monitoring
   [OK] Saved (444 chars, 7 links found)
   [PROGRESS] 6 saved, 6 visited, 2 queued
[CRAWL:1] http://localhost:8081/analytics
   [OK] Sa

KeyboardInterrupt: 

In [ ]:
# Cell 7: Save Outputs
# Save markdown files
for i, doc in enumerate(documents):
    url_path = urlparse(doc['url']).path.strip('/').replace('/', '_')
    if not url_path:
        url_path = "homepage"
    filename = f"{i:03d}_{url_path}.md"
    
    md_file = MARKDOWN_DIR / filename
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write(f"# {doc['title']}\n\n")
        f.write(f"**URL**: {doc['url']}\n\n")
        f.write(f"**Depth**: {doc['depth_level']}\n\n")
        f.write("---\n\n")
        f.write(doc['content'])

print(f"✅ Saved {len(documents)} markdown files to {MARKDOWN_DIR}")

# Save JSONL
with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f"✅ Saved JSONL corpus to {JSONL_PATH}")

✅ Saved 8 markdown files to e:\Sliit\Y4\IOT\chat_agent\data\safeguardiot_markdown
✅ Saved JSONL corpus to e:\Sliit\Y4\IOT\chat_agent\data\safeguardiot_docs.jsonl


## Quality Checks

In [ ]:
# Cell 8: Quality Checks
import random

print("[CHECK] Quality Checks:")
print()

# Check markdown files
md_files = list(MARKDOWN_DIR.glob("*.md"))
print(f"1) Markdown files: {len(md_files)}")

if len(md_files) >= 10:
    print(f"   [OK] Good coverage: {len(md_files)} pages")
elif len(md_files) >= 1:
    print(f"   [WARN] Small site crawl: {len(md_files)} pages")
else:
    raise AssertionError(f"[FAIL] No pages were saved: {len(md_files)}")

# Check JSONL
assert JSONL_PATH.exists(), "[FAIL] JSONL not found"
print()
print(f"2) JSONL file size: {JSONL_PATH.stat().st_size:,} bytes")

# Sample inspection
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    all_docs = [json.loads(line) for line in f]

samples = random.sample(all_docs, min(3, len(all_docs)))
print()
print("3) Random samples:")
print()

for i, doc in enumerate(samples, 1):
    print(f"   Sample {i}:")
    print(f"   - URL: {doc['url']}")
    print(f"   - Title: {doc['title']}")
    print(f"   - Words: {len(doc['content'].split())}")
    print()

print("[OK] All quality checks passed!")


[CHECK] Quality Checks:

1) Markdown files: 9
   [WARN] Small site crawl: 9 pages

2) JSONL file size: 11,028 bytes

3) Random samples:

   Sample 1:
   - URL: http://localhost:8081/alerts
   - Title: Safety Monitering
   - Words: 41

   Sample 2:
   - URL: http://localhost:8081/dashboard
   - Title: Safety Monitering
   - Words: 92

   Sample 3:
   - URL: http://localhost:8081/settings
   - Title: Safety Monitering
   - Words: 238

[OK] All quality checks passed!
